# LangChain Memory


<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/langchain-memory-202503.ipynb
</div>

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>

## Environment Setup

<br>

> 💡 **Important:**
>
> This notebook requires **LangChain < 0.3**.
>
> Below, you will find two options for installing the required dependencies. Choose the one that best matches your environment::
>
> 🖥️ **Running locally?**  
> → Use **Option A** to create a dedicated virtual environment (recommended).
>
> ☁️ **Using Google Colab or want a quick setup?**  
> → Use **Option B** to install the required dependencies directly.
>

<br><br>


### Option A: Use a virtual environment

Open a terminal and run the following commands.


<br>

**1. Create a virtual environment:**

```bash
python -m venv .venv/langchain-v0.2.x
```

<br>

**2. Activate the virtual environment:**

- **macOS / Linux:**
```bash
    source .venv/langchain-v0.2.x/bin/activate
```

- **Windows (PowerShell):**
```powershell
    .\.venv\langchain-v0.2.x\Scripts\Activate.ps1
```

<br>

**3. Install the required packages:**

```bash
python -m pip install \
    "langchain<0.3" \
    "langchain-core<0.3" \
    "langchain-community<0.3" \
    "langchain-openai<0.2" \
    ipykernel
```

<br>

**4. Register the environment as a Jupyter kernel:**

```bash
python -m ipykernel install \
    --user \
    --name langchain-v0.2.x \
    --display-name "Python (LangChain 0.2.x)"
```

<br>

**5. Select the correct kernel:**

Once you've completed the previous steps, do the following:
1. Open this notebook in your favourite environment (e.g., Jupyter or VS Code)
2. Select the Kernel you've just created
    - **Jupyter**: Kernel → Change Kernel → Python (LangChain 0.2.x).
    - **VS Code**: Click the Kernel selector in the top-right corner of the notebook editor → Jupyter Kernel → Python (LangChain 0.2.x)
        - Notice that you need to select "Jupyter Kernel" (not "Python Environments")
        - If "Python (LangChain 0.2.x)" doesn't appear in the kernel list, reload VS Code:
            - Press Cmd + Shift + P to open the Command Palette
            - Type Developer: Reload Window and press Enter
            - Try selecting the kernel again
3. Run the notebook as usual.

<br>

> **Notes:**
>
> - Make sure to add the directory `.venv` to your `.gitignore`
> - The virtual environment setup only needs to be completed once. The environment can then be reused for other notebooks that require **LangChain < 0.3**, without affecting your default Python environment or newer LangChain installations.

<br>

### Option B: Install dependencies directly

If you are using Google Colab, or you're having problems configuring a virtual environment, create a code cell and run the command below:

```python
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"
```

<br>


</div>

<br>

## Check LangChain version

For this notebook, you'll need LangChain 0.2.x (e.g., 0.2.17)

To check your LangChain version, you can run the command below:

In [1]:
!pip show langchain

Name: langchain
Version: 0.2.17
Summary: Building applications with LLMs through composability
Home-page: https://github.com/langchain-ai/langchain
Author: 
Author-email: 
License: MIT
Location: /Users/luis/Desktop/ironhack_june26/1_ai_eng_lectures/.venv/langchain-v0.2.x/lib/python3.11/site-packages
Requires: aiohttp, langchain-core, langchain-text-splitters, langsmith, numpy, pydantic, PyYAML, requests, SQLAlchemy, tenacity
Required-by: langchain-community


<br>

## Intro to Memory



Most common use-case for building an AI application with LangChain is a chatbot trained on a knowledgebase. But there is a problem with just using a RetrievalQA method i.e the chatbot developing using this method has no memory and thus cannot answer in a conversational manner

For example if you check the below flow

**Human** : How to add a video funnel

**AI**    : If you want to set up a video funnel, these are the steps you need to follow:

Step 1- First of all, go to the Marketing tab at the video level.

Step 2- Click on the “Add new funnel” button.

Step 3- Set the title to grab the users’ attention.

Step 4- Set the text for the action that you want would be taken by the users.

Step 5- Select the video for the Funnel.

Step 6- Set the start time at which the pop-up would appear to the user.

Step 7- Once you are satisfied with all the things, click on the “Save Changes” button.

**Human** : Now tell only 5 steps from above

**AI**.   : Sorry I am unable to answer, can you provide more context

As you can see, the chatbot has no memory and thus unable to answer any follow up questions. To solve this we will be using Memory


The memory allows a Large Language Model (LLM) to remember previous interactions with the user. By default, LLMs are stateless — meaning each incoming query is processed independently of other interactions. The only thing that exists for a stateless agent is the current input, nothing else.

There are many applications where remembering previous interactions is very important, such as chatbots. Conversational memory allows us to do that.

There are several ways that we can implement conversational memory. In the context of LangChain, they are all built on top of the `ConversationChain`.

### Types of Memory

Let's discuss some of the most popular memory methods available in Langchain.

1. ConversationBufferMemory
2. ConversationBufferWindowMemory
3. ConversationTokenBufferMemory
4. ConversationSummaryMemory

We shall discuss each of these with an example

### ConversationBufferMemory

This is a simple method that involves storing every chat interaction directly in the buffer. Although it provides good results, it has few drawbacks

1. Because every message is stored, the amount of data being sent to api is high and thus results in higher costs and slower speed of response
2. ChatGPT has input context limit which can get crossed with few messages and thus can result in error

Let's understand it's working with the help of an example. We will try adding memory to a chain

In [2]:
from langchain.llms import OpenAI
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory, ConversationBufferWindowMemory, ConversationSummaryBufferMemory

from langchain.callbacks import get_openai_callback

import os

In [3]:
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [4]:
llm = ChatOpenAI(api_key=OPENAI_API_KEY,temperature=0)

conversation = ConversationChain(
    llm=llm,
    verbose=True,
    memory=ConversationBufferMemory()
)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_73851/3250360286.py:3: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use RunnableWithMessageHistory: https://python.langchain.com/v0.2/api_reference/core/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html instead.
  conversation = ConversationChain(


We can see the prompt template used by the `ConversationChain` like so:

In [5]:
print(conversation.prompt.template)

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


Here, the prompt primes the model by telling it that the following is a conversation between a human (us) and an AI (`gpt3-turbo`). The prompt attempts to reduce hallucinations (where a model makes things up) by stating:

**"If the AI does not know the answer to a question, it truthfully says it does not know."**

This can help but does not solve the problem of hallucinations — but we will save this for the topic of a future chapter.

Following the initial prompt, we see two parameters; `{history}` and `{input}`. The `{input}` is where we’d place the latest human query; 

The `{history}` is where conversational memory is used. Here, we feed in information about the conversation history between the human and AI.

These two parameters — `{history}` and `{input}` — are passed to the LLM within the prompt template we just saw, and the output that we (hopefully) return is simply the predicted continuation of the conversation.



In [6]:
conversation_buf = ConversationChain(
    llm=llm,
    memory=ConversationBufferMemory()
)

In [7]:
conversation_buf.invoke("Good morning AI!")

{'input': 'Good morning AI!',
 'history': '',
 'response': 'Good morning! How are you today?'}

We return the first response from the conversational agent. Let’s continue the conversation, writing prompts that the LLM can only answer if it considers the conversation history. We also add a count_tokens function so we can see how many tokens are being used by each interaction.

In [8]:
def count_tokens(chain, query):
    with get_openai_callback() as cb:
        result = chain.run(query)
        print(f'Spent a total of {cb.total_tokens} tokens')

    return result

In [9]:
count_tokens(
    conversation_buf, 
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge"
)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_73851/1595156071.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  result = chain.run(query)


Spent a total of 192 tokens


"That's a fascinating topic! Large Language Models like GPT-3 have shown great potential in generating human-like text, but integrating them with external knowledge sources could greatly enhance their capabilities. There are already efforts underway to connect these models with databases, websites, and other sources of information to improve their understanding and accuracy. It's an exciting area of research with a lot of potential for innovation and advancement. Is there a specific aspect of this integration that you're particularly interested in exploring?"

In [10]:
count_tokens(
    conversation_buf,
    "I just want to analyze the different possibilities. What can you think of?"
)

Spent a total of 336 tokens


"There are several possibilities when it comes to integrating Large Language Models with external knowledge. One option is to use knowledge graphs to provide structured information that can be used to enhance the model's understanding of a given topic. Another approach is to leverage pre-trained models like BERT or RoBERTa to extract information from unstructured text and then use that information to improve the model's performance. Additionally, researchers are exploring ways to incorporate real-time data from sources like news articles or social media feeds to keep the model up-to-date and relevant. These are just a few examples of the many possibilities for integrating Large Language Models with external knowledge."

In [11]:
count_tokens(
    conversation_buf, 
    "What is my aim again?"
)

Spent a total of 399 tokens


'Your aim is to explore the potential of integrating Large Language Models with external knowledge to enhance their capabilities and improve their understanding and accuracy. You are interested in analyzing the different possibilities for this integration and understanding how it can be used to advance research and innovation in this area.'

The LLM can clearly remember the history of the conversation. Let’s take a look at how this conversation history is stored by the ConversationBufferMemory:

In [12]:
print(conversation_buf.memory.buffer)

Human: Good morning AI!
AI: Good morning! How are you today?
Human: My interest here is to explore the potential of integrating Large Language Models with external knowledge
AI: That's a fascinating topic! Large Language Models like GPT-3 have shown great potential in generating human-like text, but integrating them with external knowledge sources could greatly enhance their capabilities. There are already efforts underway to connect these models with databases, websites, and other sources of information to improve their understanding and accuracy. It's an exciting area of research with a lot of potential for innovation and advancement. Is there a specific aspect of this integration that you're particularly interested in exploring?
Human: I just want to analyze the different possibilities. What can you think of?
AI: There are several possibilities when it comes to integrating Large Language Models with external knowledge. One option is to use knowledge graphs to provide structured info

We can see that the buffer saves every interaction in the chat history directly. There are a few pros and cons to this approach. In short, they are:

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        table {
            width: 100%;
            border-collapse: collapse;
        }
        th, td {
            border: 1px solid #000;
            padding: 10px;
            text-align: left;
        }
        th {
            background-color: #f2f2f2;
        }
    </style>
</head>
<body>
    <table>
        <thead>
            <tr>
                <th>Pros</th>
                <th>Cons</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>Storing everything gives the LLM the maximum amount of information</td>
                <td>More tokens mean slowing response times and higher costs</td>
            </tr>
            <tr>
                <td>Storing everything is simple and intuitive</td>
                <td>Long conversations cannot be remembered as we hit the LLM token limit (4096 tokens for text-davinci-003 and gpt-3.5-turbo)</td>
            </tr>
        </tbody>
    </table>
</body>
</html>


The `ConversationBufferMemory` is an excellent option to get started with but is limited by the storage of every interaction. Let’s take a look at other options that help remedy this.

### ConversationSummaryMemory
Using ConversationBufferMemory, we very quickly use a lot of tokens and even exceed the context window limit of even the most advanced LLMs available today.

To avoid excessive token usage, we can use ConversationSummaryMemory. As the name would suggest, this form of memory summarizes the conversation history before it is passed to the {history} parameter.

We initialize the ConversationChain with the summary memory like so:

In [13]:
conversation_sum = ConversationChain(
	llm=llm,
	memory=ConversationSummaryMemory(llm=llm)
)

When using `ConversationSummaryMemory`, we need to pass an LLM to the object because the summarization is powered by an LLM. We can see the prompt used to do this here:

In [14]:
print(conversation_sum.memory.prompt.template)

Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.
END OF EXAMPLE

Current summary:
{summary}

New lines of conversation:
{new_lines}

New summary:


Using this, we can summarize every new interaction and append it to a “running summary” of all past interactions. Let’s have another conversation utilizing this approach.

In [15]:
# without count_tokens we'd call `conversation_sum("Good morning AI!")`
# but let's keep track of our tokens:
count_tokens(
    conversation_sum, 
    "Good morning AI!"
)

Spent a total of 256 tokens


'Good morning! How are you today?'

In [16]:
count_tokens(
    conversation_sum, 
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge"
)

Spent a total of 571 tokens


"That's a fascinating topic! Large Language Models like GPT-3 have shown great potential in generating human-like text, but integrating them with external knowledge sources could take their capabilities to the next level. There are various approaches to incorporating external knowledge, such as knowledge graphs, databases, or even real-time web scraping. It's an exciting area of research with a lot of possibilities. Do you have any specific ideas or goals in mind for this integration?"

In [17]:
count_tokens(
    conversation_sum, 
    "I just want to analyze the different possibilities. What can you think of?"
)

Spent a total of 779 tokens


"There are several ways to integrate external knowledge with Large Language Models like GPT-3. One approach is to use knowledge graphs, which organize information in a structured format that can be easily accessed by the model. Another approach is to use pre-trained embeddings that capture the relationships between words and concepts in a way that can be used to enhance the model's understanding of the world. Additionally, researchers are exploring techniques like fine-tuning the model on specific knowledge domains or using reinforcement learning to incorporate external knowledge during training. The possibilities are vast and exciting!"

In [18]:
count_tokens(
    conversation_sum, 
    "Which data source types could be used to give context to the model?"
)

Spent a total of 886 tokens


"There are several types of data sources that can be used to give context to the model. Some common ones include structured knowledge bases like Wikidata or DBpedia, unstructured text sources like Wikipedia articles or scientific papers, domain-specific databases, social media data, and even user-generated content like forums or Q&A websites. Each type of data source brings its own unique perspective and can help enrich the model's understanding of different topics and domains."

In [19]:
count_tokens(
    conversation_sum, 
    "What is my aim again?"
)

Spent a total of 883 tokens


'Your aim is to explore the potential of integrating Large Language Models with external knowledge to enhance their capabilities.'

In this case the summary contains enough information for the LLM to “remember” our original aim. We can see this summary in it’s raw form like so:

In [20]:
print(conversation_sum.memory.buffer)

The human greets the AI with a "Good morning." The AI responds with a friendly "Good morning!" and asks how the human is feeling today. The human expresses interest in exploring the potential of integrating Large Language Models with external knowledge. The AI finds this topic fascinating and discusses how integrating external knowledge sources with models like GPT-3 could enhance their capabilities. The AI mentions various approaches to incorporating external knowledge and expresses excitement about the possibilities in this area of research. The human wants to analyze the different possibilities and the AI explains various approaches like using knowledge graphs, pre-trained embeddings, fine-tuning on specific domains, and reinforcement learning to incorporate external knowledge, highlighting the vast and exciting possibilities in this field. The human asks about data source types that could be used to give context to the model, and the AI explains that structured knowledge bases, uns

The number of tokens being used for this conversation is greater than when using the `ConversationBufferMemory`, so is there any advantage to using `ConversationSummaryMemory` over the buffer memory?

<div>
<img src="https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/images-langchain-memory-rag/token_interaction.webp" alt='auto' width="1000"/>
</div>

Token count (y-axis) for the buffer memory vs. summary memory as the number of interactions (x-axis) increases.

For longer conversations. As shown above, the summary memory initially uses far more tokens. However, as the conversation progresses, the summarization approach grows more slowly. In contrast, the buffer memory continues to grow linearly with the number of tokens in the chat.

We can summarize the pros and cons of ConversationSummaryMemory as follows:

We can summarize the pros and cons of `ConversationSummaryMemory` as follows:

<!DOCTYPE html>
<html>
<head>
    <style>
        table {
            width: 100%;
            border-collapse: collapse;
        }
        th, td {
            border: 1px solid black;
            padding: 8px;
            text-align: left;
        }
        th {
            background-color: #f2f2f2;
        }
    </style>
</head>
<body>

<table>
    <tr>
        <th>Pros</th>
        <th>Cons</th>
    </tr>
    <tr>
        <td>Shortens the number of tokens for long conversations.</td>
        <td>Can result in higher token usage for smaller conversations.</td>
    </tr>
    <tr>
        <td>Enables much longer conversations.</td>
        <td>Memorization of the conversation history is wholly reliant on the summarization ability of the intermediate summarization LLM.</td>
    </tr>
    <tr>
        <td>Relatively straightforward implementation, intuitively simple to understand.</td>
        <td>Also requires token usage for the summarization LLM; this increases costs (but does not limit conversation length).</td>
    </tr>
</table>

</body>
</html>


Conversation summarization is a good approach for cases where long conversations are expected. Yet, it is still fundamentally limited by token limits. After a certain amount of time, we still exceed context window limits - Maybe NOT, depending on the LLM being used.

### ConversationBufferWindowMemory
The `ConversationBufferWindowMemor`y acts in the same way as our earlier “buffer memory” but adds a window to the memory. Meaning that we only keep a given number of past interactions before “forgetting” them. We use it like so:

In [21]:
conversation_bufw = ConversationChain(
	llm=llm,
	memory=ConversationBufferWindowMemory(k=1)
)

In this instance, we set `k=1` — this means the window will remember the single latest interaction between the human and AI. That is the latest human response and the latest AI response. We can see the effect of this below:

In [22]:
count_tokens(
    conversation_bufw, 
    "Good morning AI!"
)

Spent a total of 75 tokens


'Good morning! How are you today?'

In [23]:
count_tokens(
    conversation_bufw, 
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge"
)

Spent a total of 192 tokens


"That's a fascinating topic! Large Language Models like GPT-3 have shown great potential in generating human-like text, but integrating them with external knowledge sources could greatly enhance their capabilities. There are already efforts underway to connect these models with databases, websites, and other sources of information to improve their understanding and accuracy. It's an exciting area of research with a lot of potential for innovation and advancement. Is there a specific aspect of this integration that you're particularly interested in exploring?"

In [24]:
count_tokens(
    conversation_bufw, 
    "I just want to analyze the different possibilities. What can you think of?"
)

Spent a total of 315 tokens


"There are several possibilities when it comes to integrating Large Language Models with external knowledge. One option is to use knowledge graphs to provide structured information that can be used to enhance the model's understanding of a given topic. Another approach is to leverage pre-trained models like BERT or RoBERTa to extract information from external sources and incorporate it into the model's knowledge base. Additionally, researchers are exploring the use of reinforcement learning techniques to train models to interact with external knowledge sources in a more dynamic and adaptive way. These are just a few examples of the many possibilities for integrating Large Language Models with external knowledge."

In [25]:
count_tokens(
    conversation_bufw, 
    "Which data source types could be used to give context to the model?"
)

Spent a total of 289 tokens


'There are various data source types that can be used to give context to a model. Some common examples include text corpora, knowledge graphs, databases, structured data sources, unstructured data sources like social media posts or news articles, and domain-specific data sources. Each of these data source types can provide valuable context to help the model better understand and generate relevant information.'

In [26]:
count_tokens(
    conversation_bufw, 
    "What is my aim again?"
)

Spent a total of 214 tokens


"Your aim is to understand how different data source types can be used to provide context to a model in order to improve its performance and relevance in generating information. By utilizing various data sources effectively, you can enhance the model's understanding and output in a specific domain or task."

By the end of the conversation, when we ask **"What is my aim again?"**, the answer to this was contained in the human response three interactions ago. As we only kept the most recent interaction (`k=1`), the model had forgotten and could not give the correct answer.

We can see the effective “memory” of the model like so:

In [27]:
bufw_history = conversation_bufw.memory.load_memory_variables(
    inputs=[]
)['history']

In [28]:
print(bufw_history)

Human: What is my aim again?
AI: Your aim is to understand how different data source types can be used to provide context to a model in order to improve its performance and relevance in generating information. By utilizing various data sources effectively, you can enhance the model's understanding and output in a specific domain or task.


Although this method isn’t suitable for remembering distant interactions, it is good at limiting the number of tokens being used — a number that we can increase/decrease depending on our needs. For the longer conversation used in our earlier comparison, we can set `k=6` and reach ~1.5K tokens per interaction after 27 total interactions:

<div>
<img src="https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/images-langchain-memory-rag/conversation_bw.webp" alt='auto' width="1000"/>
</div>

Token count including the ConversationBufferWindowMemory at k=6 and k=12.

If we only need memory of recent interactions, this is a great option. However, for a mix of both distant and recent interactions, there are other options.

### ConversationSummaryBufferMemory
The `ConversationSummaryBufferMemory` is a mix of the `ConversationSummaryMemory` and the `ConversationBufferWindowMemory`. It summarizes the earliest interactions in a conversation while maintaining the max_token_limit most recent tokens in their conversation. It is initialized like so:

In [29]:
conversation_sum_bufw = ConversationChain(
    llm=llm, memory=ConversationSummaryBufferMemory(
        llm=llm,
        max_token_limit=650
))

When applying this to our earlier conversation, we can set `max_token_limit` to a small number and yet the LLM can remember our earlier “aim”.

This is because that information is captured by the “summarization” component of the memory, despite being missed by the “buffer window” component.

Naturally, the pros and cons of this component are a mix of the earlier components on which this is based.

<!DOCTYPE html>
<html>
<head>
    <style>
        table {
            width: 100%;
            border-collapse: collapse;
        }
        th, td {
            border: 1px solid black;
            padding: 10px;
            text-align: left;
        }
        th {
            background-color: #f2f2f2;
        }
    </style>
</head>
<body>

<table>
    <tr>
        <th>Pros</th>
        <th>Cons</th>
    </tr>
    <tr>
        <td>Summarizer means we can remember distant interactions</td>
        <td>Summarizer increases token count for shorter conversations</td>
    </tr>
    <tr>
        <td>Buffer prevents us from missing information from the most recent interactions</td>
        <td>Storing the raw interactions — even if just the most recent interactions — increases token count</td>
    </tr>
</table>

</body>
</html>


Although requiring more tweaking on what to summarize and what to maintain within the buffer window, the `ConversationSummaryBufferMemory` does give us plenty of flexibility and is the only one of our memory types (so far) that allows us to remember distant interactions and store the most recent interactions in their raw — and most information-rich — form.

<div>
<img src="https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/images-langchain-memory-rag/memory_bws.webp" alt='auto' width="1000"/>
</div>

Token count comparisons including the ConversationSummaryBufferMemory type with max_token_limit values of 650 and 1300.

We can also see that despite including a summary of past interactions and the raw form of recent interactions — the increase in token count of `ConversationSummaryBufferMemory` is competitive with other methods.

### Other Memory Types
The memory types we have covered here are great for getting started and give a good balance between remembering as much as possible and minimizing tokens.

However, we have other options — particularly the `ConversationKnowledgeGraphMemory` and `ConversationEntityMemory`. We’ll give these different forms of memory the attention they deserve in upcoming chapters.

That’s it for this introduction to conversational memory for LLMs using LangChain. As we’ve seen, there are plenty of options for helping stateless LLMs interact as if they were in a stateful environment — able to consider and refer back to past interactions.

As mentioned, there are other forms of memory we can cover. We can also implement our own memory modules, use multiple types of memory within the same chain, combine them with agents, and much more. All of which we will cover in the future.